In [1]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("blastchar/telco-customer-churn")

df = pd.read_csv(path + "/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.head())
print(df.shape)

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 172k/172k [00:00<00:00, 176kB/s]

Extracting files...
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies 

In [2]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

C:\Users\gurra\AppData\Local\Temp\ipykernel_18844\3920077276.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [3]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X = pd.get_dummies(X, drop_first=True)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
from sklearn.ensemble import RandomForestClassifier

churn_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42
)

churn_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=50, random_state=42)

In [7]:
from sklearn.ensemble import RandomForestRegressor

y_reg = df["MonthlyCharges"]

X_reg = X

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42
)

rev_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42
)

rev_model.fit(X_train_r, y_train_r)

RandomForestRegressor(n_estimators=50, random_state=42)

In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)

df["Cluster"] = kmeans.fit_predict(X)

In [9]:
def retention_recommendation(churn_prob):
    if churn_prob > 80:
        return "Offer 30% discount + priority support"
    elif churn_prob > 50:
        return "Provide loyalty rewards"
    else:
        return "Maintain regular engagement"

In [10]:
import pickle

pickle.dump(churn_model, open("churn_model.pkl", "wb"))
pickle.dump(rev_model, open("revenue_model.pkl", "wb"))
pickle.dump(scaler, open("churn_scaler.pkl", "wb"))
pickle.dump(X.columns, open("churn_columns.pkl", "wb"))
pickle.dump(kmeans, open("cluster_model.pkl", "wb"))

In [11]:
import json
import pandas as pd
import pickle

model = pickle.load(open("churn_model.pkl", "rb"))
rev_model = pickle.load(open("revenue_model.pkl", "rb"))
scaler = pickle.load(open("churn_scaler.pkl", "rb"))
columns = pickle.load(open("churn_columns.pkl", "rb"))

def lambda_handler(event, context):

    df = pd.DataFrame([event])
    df = pd.get_dummies(df)
    df = df.reindex(columns=columns, fill_value=0)

    scaled = scaler.transform(df)

    churn_prob = model.predict_proba(scaled)[0][1] * 100
    revenue_loss = rev_model.predict(df)[0]

    recommendation = retention_recommendation(churn_prob)

    return {
        "statusCode": 200,
        "body": json.dumps({
            "churn_probability": round(churn_prob, 2),
            "expected_revenue_loss": round(revenue_loss, 2),
            "recommendation": recommendation
        })
    }